# Chapter 1 — Types are values

Step 1 builds the vocabulary every cache key will be written in.

**Source built this step** (with counted-line budgets from `scripts/loc_budget.py`):

| File | Counted lines / cap | What |
|---|---|---|
| `src/pdum/dsl/kernel/types.py` | 94 / 100 | the structural `Type` lattice + `TemplateId` code identity |
| `src/pdum/dsl/kernel/valuekind.py` | 61 / 95 | `typeof` + `fingerprint` (the marshaling half arrives in ch08) |

Tests: `tests/test_types.py`, `tests/test_valuekind.py` (including the
soundness fuzz), `tests/test_budgets.py`. Glossary terms introduced: *Type,
summary function, typeof, fingerprint, ValueKind, TemplateId, FnType, Literal
lift, Tuple-vs-Vec* — see [GLOSSARY.md](GLOSSARY.md).

In [1]:
from pdum.dsl.kernel import types as T

color = T.Record("Color", (("r", T.f32), ("g", T.f32), ("b", T.f32)))
examples = [
    T.f64,
    T.Tuple((T.f64, T.f64)),
    T.Vec(T.f32, 3),
    T.Array(T.f32, 2, "C", "<", True),
    color,
    T.LiteralType(T.i64, 8),
    T.Array(T.f32, 2, "C", "<", False),
]
for t in examples:
    print(f"{t!r:<40} hash={hash(t) & 0xFFFF:04x}")

f64                                      hash=9fb2
(f64, f64)                               hash=2a15
vec3<f32>                                hash=91f1
array<f32,2d,C>                          hash=a3d8
Color{r: f32, g: f32, b: f32}            hash=3a47
Literal[i64 = 8]                         hash=3c6b
array<f32,2d,C,ro>                       hash=3c7f


A `Type` is a **frozen dataclass compared structurally** — two independently
constructed types are equal, hash equal, and collide in a dict on purpose.
That property is the entire reason they can serve as cache keys. Note
`LiteralType`: the *one* place a value is allowed inside a type, and only by
explicit opt-in (the future `Literal` lift — bake-as-constant, recompile per
value).

In [2]:
import dataclasses

print("equal?     ", T.Scalar("f64") == T.f64)
print("same obj?  ", T.Scalar("f64") is T.f64)

artifact_cache = {T.Vec(T.f32, 3): "<pretend compiled pipeline>"}
print("dict probe:", artifact_cache[T.Vec(T.f32, 3)])

# Literal lifts are TYPE-AWARE about their value: Python's == is cross-type
# (1 == 1.0 == True), which would merge distinct baked constants into one key.
print("Literal 1 vs 1.0:", T.LiteralType(T.f64, 1) == T.LiteralType(T.f64, 1.0))

try:
    T.f64.kind = "f32"
except dataclasses.FrozenInstanceError as e:
    print("frozen:    ", e)

equal?      True
same obj?   False
dict probe: <pretend compiled pipeline>
Literal 1 vs 1.0: False
frozen:     cannot assign to field 'kind'


In [3]:
# Validation is loud at construction — a malformed type must never reach a key.
for build in (lambda: T.Scalar("f16"), lambda: T.Vec(T.f32, 5), lambda: T.Vec(T.Vec(T.f32, 2), 2)):
    try:
        build()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

ValueError: unknown scalar kind 'f16'; expected one of ['bool', 'f32', 'f64', 'i32', 'i64', 'u32', 'u64']
ValueError: Vec length must be 2..4, got 5
TypeError: Vec element must be a Scalar, got vec2<f32>


## `typeof` is a summary function, not a class lookup

The question "what is this value's type" is answered per-Python-type by a
registered **ValueKind**, and the kind *chooses* how much of the value its
summary records (architecture §13). The built-in int kind is the canonical
example: it reads the value's **range** — `5` and `2**63` get different
types, because packing them into the same 8 bytes would corrupt one of them.
Later, an opt-in array kind will put *shapes* in the type the same way. The
dial exists per kind; nothing is global.

In [4]:
from pdum.dsl.kernel.valuekind import BigIntError, fingerprint, typeof

print(typeof(True), "|", typeof(5), "|", typeof(1.5))
print(typeof(2**63 - 1), "vs", typeof(2**63), "  <- same Python class, different summaries")
try:
    typeof(2**70)
except BigIntError as e:
    print("BigIntError:", e)

# Unregistered types are LOUD — guessing would put an unsound key in the cache.
try:
    typeof(object())
except TypeError as e:
    print("TypeError:  ", e)

bool | i64 | f64
i64 vs u64   <- same Python class, different summaries
BigIntError: int 1180591620717411303424 does not fit in 64 bits
TypeError:   no ValueKind registered for 'object'; register one (or give the class a __dsl_type__) rather than guessing


In [5]:
# Python tuples summarize ELEMENT-WISE to Tuple — honestly, including
# heterogeneous and nested ones. This is the `center = (cx, cy)` capture that
# M0 could not express, with no vec interpretation smuggled in.
print(typeof((1.0, 2.0)), "|", typeof((1, 2.0)), "|", typeof(((1.0, 2.0), 3)))
print("arity is part of the identity:", typeof((1, 2)) != typeof((1, 2, 3)))

# Loudness comes only from elements:
try:
    typeof((1.0, 2**70))
except BigIntError as e:
    print("an element error surfaces:", e)

(f64, f64) | (i64, f64) | ((f64, f64), i64)
arity is part of the identity: True
an element error surfaces: int 1180591620717411303424 does not fit in 64 bits


> **Resolved at this chapter's walkthrough.** An earlier draft of this step
> summarized homogeneous scalar tuples as `Vec` — a shader-language
> interpretation leaking into the identity layer. The rule is now: **`typeof`
> produces `Tuple`, never `Vec`.** `Vec` stays in the lattice purely as an
> *IR-level* type that dialect lowering rules produce (a 3-tuple literal in a
> shader's return position becomes `core.vec`), and whether a captured
> `Tuple((f64, f64))` is *packed* as one `vec2<f32>` uniform or two scalar
> slots is the backend's PackPlan decision (ch08/ch10). This is the same split
> M0 documented as "two type levels" — the ledger entry lives in
> `docs/design/010_proposed-architecture.md` §10, and this note stays in the book as
> provenance of the correction.

In [6]:
import timeit

v = (1.0, 2.5, 3.5)
print("fingerprint:", fingerprint(v))
print("typeof:     ", typeof(v))

n = 100_000
def best(fn):  # min-of-repeats is standard timeit practice; single runs have ±30% noise
    return min(timeit.repeat(lambda: fn(v), number=n, repeat=5)) / n * 1e9
print(f"fingerprint {best(fingerprint):6.0f} ns/call   typeof {best(typeof):6.0f} ns/call   (best of 5)")

fingerprint: ('t', ('f64', 'f64', 'f64'))
typeof:      (f64, f64, f64)


fingerprint    746 ns/call   typeof    831 ns/call   (best of 5)


The **fingerprint** is the hot-path stand-in for `typeof`: it must be cheap
(it runs on every call in the future dispatch path) and it is bound by the
**soundness law** — `fingerprint(a) == fingerprint(b)` must imply the same
`typeof` outcome. A fingerprint collision would be a *silent wrong cache
hit*: the worst failure class in the whole system, worse than a crash.

Notice the two calls above measure **nearly the same** today — expected, not
alarming. For scalar-family kinds the summary *is* cheap: interned singletons
plus one frozen-dataclass wrap. The fingerprint earns its keep on kinds where
building the full `Type` is real work — the array kind (`typeof` reads
dtype/flags/layout and constructs an `Array`; the fingerprint is a small
precomputed tag) and the `Handle` kind (`typeof` builds an `FnType`) — which
arrive in later chapters. Capture fingerprints are also computed once at
phase A and memoized on the `Handle`; only *argument* fingerprints run per
call. The step-9 microbenchmark gate decides these trade-offs with data — one
live option is `fingerprint := the Type itself` for cheap kinds, making
soundness trivially true.

The soundness law is enforced by a property fuzz in CI; here it is, live:

In [7]:
import random

from pdum.dsl.kernel import types as T
from pdum.dsl.kernel.valuekind import BUILTINS


def soundness_fuzz(table, n=2000, seed=20260711):
    rng = random.Random(seed)

    def value(depth=0):
        r = rng.random()
        if r < 0.15:
            return rng.choice([True, False])
        if r < 0.45:
            span = rng.choice([2**8, 2**62, 2**63, 2**64, 2**66])
            return rng.randrange(-span, span)
        if r < 0.7 or depth >= 2:
            return rng.uniform(-1e9, 1e9)
        return tuple(value(depth + 1) for _ in range(rng.randrange(0, 6)))

    # (This generator has a CI twin in tests/test_valuekind.py — grow BOTH
    # when the value universe grows.)
    def outcome(fn, v):
        try:
            return ("ok", fn(v))
        except TypeError as e:  # BigIntError subclasses TypeError
            return ("raise", type(e).__name__)

    seen = {}
    for _ in range(n):
        v = value()
        fp, ty = outcome(table.fingerprint, v), outcome(table.typeof, v)
        if fp[0] == "raise":
            assert ty[0] == "raise", f"fingerprint raised but typeof didn't for {v!r}"
            continue
        prior = seen.setdefault(fp, (v, ty))
        if prior[1] != ty:
            return f"SOUNDNESS VIOLATION: {prior[0]!r} and {v!r} share {fp} but differ: {prior[1]} vs {ty}"
    return f"sound over {n} random values"


print(soundness_fuzz(BUILTINS))

sound over 2000 random values


In [8]:
# Now break the law on purpose: a float kind whose fingerprint collides with
# ints. Kinds receive the dispatching TABLE on every call — that is what lets
# BUILTINS.extend() overrides reach elements nested inside tuples too.
class SloppyFloatKind:
    def typeof(self, v, table):
        return T.f64

    def fingerprint(self, v, table):
        return "i64"  # deliberately wrong: collides with the int bucket tag

    def flatten(self, v, table):  # the marshaling view (ch08); part of the contract
        return (v,)


sloppy = BUILTINS.extend()  # a layered child table; BUILTINS itself is untouched
sloppy.register(float, SloppyFloatKind())
print("override reaches nested floats:", sloppy.fingerprint((1.5, 2.5)))

print(soundness_fuzz(sloppy))

override reaches nested floats: ('t', ('i64', 'i64'))
SOUNDNESS VIOLATION: 33262194.91420865 and -6845907341870914099 share ('ok', 'i64') but differ: ('ok', f64) vs ('ok', i64)


That violation, in the real system, would have been `closure(1.5)` silently
reusing the artifact compiled for `closure(1)` — packing a float into an i32
uniform slot. The fuzzer exists so that enriching a type (say, adding shapes)
while forgetting the fingerprint fails CI instead of corrupting a buffer.

In [9]:
# Code identity: CPython compares code objects BY VALUE. This single fact is
# the live-coding story — an unchanged re-run hits the cache, an edit misses.
SRC = "def f(x):\n    return x + k\n"


def code_of(src):
    ns = {}
    exec(compile(src, "<notebook-cell>", "exec"), ns)
    return ns["f"].__code__


a, b = code_of(SRC), code_of(SRC)
print("same source, two compiles:   a is b ->", a is b, "  a == b ->", a == b)
print("after a one-token edit:      ", a == code_of("def f(x):\n    return x + k + 1\n"))

base = T.Base(a)
grad = T.Derived("grad", base, (("wrt", 0),))
print(base, "|", grad, "| collide? ->", base == grad)

# FnType — the thesis in one value (ch02 will build these from real closures):
print(T.FnType(base, (T.i64,)), "  rebuilt-equal ->", T.FnType(T.Base(b), (T.i64,)) == T.FnType(base, (T.i64,)))

same source, two compiles:   a is b -> False   a == b -> True
after a one-token edit:       False
Base<f> | Derived<grad(f)> | collide? -> False
Fn<f>(i64)   rebuilt-equal -> True


## What we can't do yet

- `typeof(None)`, arrays, records, `Handle`s — kinds arrive with their users
  (arrays/records in the stdlib around ch11–11, `Handle` in ch02).
- Nothing *produces* an `FnType` from a real closure yet — that is phase-A
  capture, **ch02**.
- `leaf_types`/`flatten` (the marshaling views of a `ValueKind`) — ch08.
- The kind table is a module-level builtin seed; it folds into the explicit
  `Registry` (surface C) in ch09.

**Budget after step 1:** kernel 155 / 1150 counted lines.

**Next:** ch02 — what a closure is: `make_handle`, snapshots, and `FnType`s
built from live closure cells.